# SLR-RAG : Step 1

## Package import

In [1]:
# Import packages
%load_ext autoreload
%autoreload 2
import pandas as pd
from slrrag.querier_scopus import query_scopus

pd.__version__

'3.0.3'

## Inputs

### Query

### Reference papers
Represent them as (TITLE,DOI)
The notebook will try to do the merge based on the DOI first, and on the title otherwise if the paper has no DOI.

In [2]:
from slrrag.main import get_new_reference_df

df_reference_paper = get_new_reference_df()
df_reference_paper

,title,doi,type,year,date,abstract
0,A dynamic risk assessment method to address sa...,10.1177/1475090219837409,NaN,<NA>,NaT,NaN
1,A Marine GIS-based Alert System to Prevent Ves...,10.51400/2709-6998.2462,NaN,<NA>,NaT,NaN
2,A method for risk analysis of ship collisions ...,10.1016/j.oceaneng.2021.109396,NaN,<NA>,NaT,NaN
3,A Novel Method for Navigational Risk Assessmen...,10.1155/2021/4588333,NaN,<NA>,NaT,NaN
4,A System-Theory and Complex Network-Fused Appr...,10.3390/jmse11071306,NaN,<NA>,NaT,NaN
5,Allision risk analysis of offshore petroleum i...,10.1007/s13437-016-0123-7,NaN,<NA>,NaT,NaN
6,Assessing the impacts to vessel traffic from o...,NaN,NaN,<NA>,NaT,NaN
7,Assessing the validity of navigation risk asse...,10.1016/j.ocecoaman.2022.106078,NaN,<NA>,NaT,NaN
8,Evaluation of Ship Importance in Offshore Wind...,10.3390/su17188252,NaN,<NA>,NaT,NaN
9,Improving the coexistence of offshore wind far...,10.1007/s13437-018-0149-0,NaN,<NA>,NaT,NaN


## Query databases

### Add scopus papers

In [3]:
from slrrag.settings import PATH_CSV_04_MANUAL_FILTER
from slrrag.toolbox.files import slugify

df_04_check = pd.read_csv(PATH_CSV_04_MANUAL_FILTER)
df_04_check["title_slug"] = df_04_check["title"]  # .apply(slugify)
df_04_check = df_04_check[
    (~df_04_check["id_scopus"].isna()) & (df_04_check["tags"] == "#keep")
]

df_04_check.head()

,tags,title,accessed,doi,id_lens,id_scopus,id_openalex,id_dimensions,pdf,title_slug
24,#keep,a-collision-risk-based-method-for-safety-dista...,Yes,10.1016/j.ress.2026.112565,NaN,2-s2.0-105032710553,https://openalex.org/W7134194449,https://app.dimensions.ai/details/publication/...,https://content.openalex.org/works/W7134194449...,a-collision-risk-based-method-for-safety-dista...
25,#keep,a-colreg-compliant-ship-collision-alert-system...,Yes,10.1016/j.oceaneng.2020.107866,035-702-261-461-03X,2-s2.0-85088825486,https://openalex.org/W3092111448,https://app.dimensions.ai/details/publication/...,https://content.openalex.org/works/W3092111448...,a-colreg-compliant-ship-collision-alert-system...
54,#keep,a-data-driven-approach-for-determining-navigat...,Yes,10.1016/j.oceaneng.2026.124509,005-788-810-540-155,2-s2.0-105032113578,https://openalex.org/W7128947758,https://app.dimensions.ai/details/publication/...,https://content.openalex.org/works/W7128947758...,a-data-driven-approach-for-determining-navigat...
55,#keep,a-data-driven-approach-for-ship-bridge-collisi...,Yes,10.1016/j.oceaneng.2022.113137,NaN,2-s2.0-85142360588,https://openalex.org/W4310014528,https://app.dimensions.ai/details/publication/...,https://content.openalex.org/works/W4310014528...,a-data-driven-approach-for-ship-bridge-collisi...
57,#keep,a-data-driven-framework-for-detecting-unsafe-s...,Yes,10.3390/jmse14100944,NaN,2-s2.0-105040214376,https://openalex.org/W7162185041,https://app.dimensions.ai/details/publication/...,https://content.openalex.org/works/W7162185041...,a-data-driven-framework-for-detecting-unsafe-s...


In [4]:
from slrrag.main import add_is_reference_column, NAME_COL_IS_REFERENCE

main_query = """
(
    (
        ("offshore") 
        AND 
        ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR  "windmill" OR 
        "wind turbine" OR "wind park" OR "wind energy" OR "platform" OR "quay" OR "wharf" OR "lidar")
    )
    OR 
    (("bridge" AND NOT "bridge the gap" AND NOT "bridge therapy") OR "flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")
)

AND

(
    ("Navigation risk assessment" OR "Navigational risk assessment" OR "Navigation risk assessments" OR "Navigational risk assessments") 
    OR
    (
        ("collision" OR "allision" OR (("risk" OR "safe" OR "safety") AND ("traffic" OR "shipping"))) 
        AND 
        ("ship" OR ("vessel" AND NOT "blood vessel")) 
        AND 
        ("probability" OR "probabilistic" OR "likelihood" OR "chance" OR "assess" OR "assessing" OR 
        "assessment" OR "model" OR "modeling" OR "modelisation" OR "predict" OR "predicting" OR 
        "prediction" OR "analyse" OR "analysing" OR "analyze" OR "analyzing" OR "study" OR "studies" OR "studying" OR "detect" OR "detection")
    )
)

"""

# You can edit your scopus API key in ~/.config/pybliometrics.cfg if needed

is_dry_run = False
df_scopus = query_scopus(main_query, dry_run=is_dry_run)
if not is_dry_run:
    # Check if reference papers are present from previous query on scopus
    df_scopus = add_is_reference_column(df_scopus)
    print("---\nReference papers in Scopus : ", df_scopus[NAME_COL_IS_REFERENCE].sum())

1975 found in Scopus.
---
Reference papers in Scopus :  19


In [5]:
# Check if all papers are present from previous query on scopus
df_04_check[~(df_04_check["id_scopus"].isin(df_scopus["id_scopus"]))]["title"].values

<StringArray>
[                                                                                     'a-data-driven-approach-for-determining-navigation-waters-influenced-by-bridges-a-framework-and-a-case-study',
                                                                                         'a-monocular-vision-based-safety-monitoring-framework-for-offshore-infrastructures-utilizing-grounded-sam',
                                                          'a-novel-fuzzy-bayesian-network-based-madm-model-for-offshore-wind-turbine-selection-in-busy-waterways-an-application-to-a-case-in-china',
                                                                                       'a-path-planning-approach-based-on-multi-direction-a-algorithm-for-ships-navigating-within-wind-farm-waters',
                                                                                                                                           'acceptable-risk-criteria-for-infrastructure-protection',
 

In [6]:
from slrrag.main import merge_papers

df_papers_01 = merge_papers(df_reference_paper, df_scopus, "scopus")
df_papers_01

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,is_reference
0,10th International Conference on Coastal and O...,NaN,other,<NA>,2024-01-01,The proceedings contain 28 papers. The special...,2-s2.0-85200940949,NaN,554,False
1,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886697517,NaN,361,False
2,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886707330,NaN,362,False
3,"12th AIAA Aviation Technology, Integration, an...",NaN,other,<NA>,2012-12-01,The proceedings contain 306 papers. The specia...,2-s2.0-84880934476,NaN,1042,False
4,"13th International Marine Design Conference, I...",NaN,other,<NA>,2018-01-01,The proceedings contain 111 papers. The specia...,2-s2.0-85061337310,NaN,458,False
...,...,...,...,...,...,...,...,...,...,...
1972,WOE-VesselTrack: Robust Vessel Position Measur...,10.1109/TIM.2026.3654750,article,<NA>,2026-01-01,Real-time monitoring of vessel navigation cond...,2-s2.0-105027972882,Adaptive anti-occlusion | deep learning | imag...,1120,False
1973,"Work at sea: a study of sleep, and of circadia...",10.1007/BF00405665,article,<NA>,1988-05-01,The safety of a ship depends substantially on ...,2-s2.0-0023949876,Circadian rhythms | Fatigue | Performance effi...,1030,False
1974,Work-Life Conditions as the Primary Determinan...,10.1177/00469580261438708,article,<NA>,2026-01-01,Seafarer mental health has become an increasin...,2-s2.0-105035570719,machine learning | random forest | seafarer me...,1587,False
1975,WTMS - Waterway traffic management system - A ...,10.1061/40555(2001)126,conference-paper,<NA>,2004-12-01,Waterways are well known as an efficient and r...,2-s2.0-74949088209,NaN,1146,False


### Add OpenAlex papers

You will need the API key of

Make sure you have the API key of OpenAlex setup.

In [8]:
from slrrag.querier_openalex import query_openalex

df_openalex = query_openalex(main_query)
df_openalex

Query : 
(
    (
        ("offshore") 
        AND 
        ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR  "windmill" OR 
        "wind turbine" OR "wind park" OR "wind energy" OR "platform" OR "quay" OR "wharf" OR "lidar")
    )
    OR 
    (("bridge" AND NOT "bridge the gap" AND NOT "bridge therapy") OR "flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")
)

AND

(
    ("Navigation risk assessment" OR "Navigational risk assessment" OR "Navigation risk assessments" OR "Navigational risk assessments") 
    OR
    (
        ("collision" OR "allision" OR (("risk" OR "safe" OR "safety") AND ("traffic" OR "shipping"))) 
        AND 
        ("ship" OR ("vessel" AND NOT "blood vessel")) 
        AND 
        ("probability" OR "probabilistic" OR "likelihood" OR "chance" OR "assess" OR "assessing" OR 
        "assessment" OR "model" OR "modeling" OR "modelisation" OR "predict" OR "predicting" OR 
        "prediction" OR "analyse"

 99%|█████████▊| 48/48.72 [00:56<00:00,  1.03it/s]/home/vlitoux/Documents/SLR-RAG/.venv/lib/python3.12/site-packages/tqdm/std.py:635: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
101%|██████████| 49/48.72 [00:58<-00:00,  1.19s/it]


,doi,id_openalex,title,date,abstract,type,year,relevance_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex
0,10.1016/j.ocecoaman.2022.106078,https://openalex.org/W4211011661,Assessing the validity of navigation risk asse...,2022-02-09,The developments of offshore wind farms can pl...,article,<NA>,597.286400,https://content.openalex.org/works/W4211011661...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Risk assessment|Risk analysis (engineering)|Cr...
1,10.1016/j.ocecoaman.2022.106428,https://openalex.org/W4310345253,A novel fuzzy multi-factor navigational risk a...,2022-11-24,NaN,article,<NA>,538.583600,https://content.openalex.org/works/W4310345253...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Analytic hierarchy process|Offshore wind power...
2,10.1016/j.ress.2020.107086,https://openalex.org/W3039653437,Realising advanced risk assessment of vessel t...,2020-07-02,NaN,article,<NA>,538.188540,https://content.openalex.org/works/W3039653437...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Offshore wind power|Turbine|Bayesian network|C...
3,10.1016/j.ress.2012.02.008,https://openalex.org/W2080697854,Influences of variables on ship collision prob...,2012-03-06,NaN,article,<NA>,512.113800,https://content.openalex.org/works/W2080697854...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Bayesian network|Bridge (graph theory)|Variabl...
4,10.1137/1114019,https://openalex.org/W2137295153,Non-Parametric Estimation of a Multivariate Pr...,1969-01-01,Previous article Next article Non-Parametric E...,article,<NA>,491.318000,https://content.openalex.org/works/W2137295153...,Statistical Methods and Inference,Mathematics,Statistics and Probability,Physical Sciences,Multivariate statistics|Mathematics|Multivaria...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2437,NaN,https://openalex.org/W2756303712,Huntington Beach narrative end item report - S...,1969-01-01,"tion paragraph number, the HCCO drawing nrnnbe...",article,<NA>,0.233254,https://content.openalex.org/works/W2756303712...,Underground infrastructure and sustainability,Engineering,Building and Construction,Physical Sciences,Saturn|Narrative|Huntington's disease|History|...
2438,10.22541/au.175527531.14142280/v1,https://openalex.org/W4413468196,Knill’s Magic 2.o: The Underworlding Conflict ...,2025-08-15,Knill’s Magic 2.o: Underworlding War and Water...,preprint,<NA>,0.215244,https://content.openalex.org/works/W4413468196...,Themes in Literature Analysis,Arts and Humanities,Literature and Literary Theory,Social Sciences,MAGIC (telescope)|Transformation (genetics)|Po...
2439,10.22541/au.175527531.14142280/v2,https://openalex.org/W4413783209,Knill’s Magic 2.o: Underworlding Across War an...,2025-08-28,Knill’s Magic 2.o: Underworlding War and Water...,preprint,<NA>,0.213828,https://content.openalex.org/works/W4413783209...,Themes in Literature Analysis,Arts and Humanities,Literature and Literary Theory,Social Sciences,MAGIC (telescope)|Political science|Physics|As...
2440,10.22541/au.175527531.14142280/v3,https://openalex.org/W4414608372,Knill’s Magic 2.o: Underworlding War and Water,2025-09-29,Knill’s Magic 2.o: Underworlding War and Water...,preprint,<NA>,0.213480,https://content.openalex.org/works/W4414608372...,Themes in Literature Analysis,Arts and Humanities,Literature and Literary Theory,Social Sciences,MAGIC (telescope)|Bureaucracy|Impulse (physics...


In [13]:
from slrrag.main import merge_papers

df_papers_02 = merge_papers(df_papers_01, df_openalex, "openalex")
df_papers_02

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,is_reference,id_openalex,relevance_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex
0,10th International Conference on Coastal and O...,NaN,other,<NA>,2024-01-01,The proceedings contain 28 papers. The special...,2-s2.0-85200940949,NaN,554,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886697517,NaN,361,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886707330,NaN,362,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"12th AIAA Aviation Technology, Integration, an...",NaN,other,<NA>,2012-12-01,The proceedings contain 306 papers. The specia...,2-s2.0-84880934476,NaN,1042,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"13th International Marine Design Conference, I...",NaN,other,<NA>,2018-01-01,The proceedings contain 111 papers. The specia...,2-s2.0-85061337310,NaN,458,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3313,파일지지 구조물의 선박 충돌거동에 대한 해석,NaN,article,<NA>,2008-05-01,The ship collision analysis of steel pile grou...,NaN,NaN,<NA>,False,https://openalex.org/W1548055064,2.374983,https://content.openalex.org/works/W1548055064...,Engineering Applied Research,Engineering,Civil and Structural Engineering,Physical Sciences,Collision|Pier|Pile|Pile cap|Structural engine...
3314,평택·당진항의 내항개발에 따른 교통혼잡도 평가에 관한 연구,NaN,article,<NA>,2005-08-31,The port entry system of the inner harbor in P...,NaN,NaN,<NA>,False,https://openalex.org/W3183424587,1.518602,https://content.openalex.org/works/W3183424587...,Maritime Ports and Logistics,Engineering,Industrial and Manufacturing Engineering,Physical Sciences,Port (circuit theory)|Bridge (graph theory)|Ma...
3315,항만횡단 해상교량의 해양사고 관련 인자 분석을 통한 교량설계안 제안,NaN,article,<NA>,2008-12-31,Recently Bridges crossing waterway are constru...,NaN,NaN,<NA>,False,https://openalex.org/W2476308561,1.719204,https://content.openalex.org/works/W2476308561...,Marine and Coastal Research,Engineering,Ocean Engineering,Physical Sciences,Bridge (graph theory)|Span (engineering)|Mista...
3316,해상교량에 대한 선박충돌 위험도 평가 및 민감도 분석,NaN,article,<NA>,2013-09-01,In the design phase of sea-crossing bridge pro...,NaN,NaN,<NA>,False,https://openalex.org/W2197667956,2.922672,https://content.openalex.org/works/W2197667956...,Engineering Applied Research,Engineering,Civil and Structural Engineering,Physical Sciences,Collision|Bridge (graph theory)|Risk assessmen...


### Add dimensions

In [24]:
from slrrag.main import print_dimensions_export
from slrrag.query_converter import query_to_dimensions_url
from slrrag.settings import PATH_DIMENSIONS_EXPORT


dimensions_url = query_to_dimensions_url(main_query)
export_path_dimensions = PATH_DIMENSIONS_EXPORT
print_dimensions_export(dimensions_url, export_path_dimensions)


Dimensions.ai URL:
https://app.dimensions.ai/discover/publication?search_mode=content&search_text=%0A(%0A%20%20%20%20(%0A%20%20%20%20%20%20%20%20(%22offshore%22)%20%0A%20%20%20%20%20%20%20%20AND%20%0A%20%20%20%20%20%20%20%20(%22infrastructure%22%20OR%20%22construction%22%20OR%20%22installation%22%20OR%20%22windfarm%22%20OR%20%22wind%20farm%22%20OR%20%20%22windmill%22%20OR%20%0A%20%20%20%20%20%20%20%20%22wind%20turbine%22%20OR%20%22wind%20park%22%20OR%20%22wind%20energy%22%20OR%20%22platform%22%20OR%20%22quay%22%20OR%20%22wharf%22%20OR%20%22lidar%22)%0A%20%20%20%20)%0A%20%20%20%20OR%20%0A%20%20%20%20((%22bridge%22%20AND%20NOT%20%22bridge%20the%20gap%22%20AND%20NOT%20%22bridge%20therapy%22)%20OR%20%22flotel%22%20OR%20%22Aquaculture%20farm%22%20OR%20%22Artificial%20Island%22%20OR%20%22Synthetic%20Island%22)%0A)%0A%0AAND%0A%0A(%0A%20%20%20%20(%22Navigation%20risk%20assessment%22%20OR%20%22Navigational%20risk%20assessment%22%20OR%20%22Navigation%20risk%20assessments%22%20OR%20%22Navigation

In [28]:
from slrrag.querier_dimensions import read_dimensions_export


df_dimensions = read_dimensions_export(export_path_dimensions)
df_dimensions

,doi,title,date,abstract,type,year,id_dimensions,topic_dimensions,keywords_dimensions
0,10.1016/j.marpolbul.2014.06.004,Getting it right for the North Atlantic right ...,2014-07-04,The North Atlantic right whale (Eubalaena glac...,article,<NA>,https://app.dimensions.ai/details/publication/...,"C13 Architecture, Built Environment and Planning",Animal Migration; Animals; Atlantic Ocean; Eco...
1,10.3390/s20051246,Safe Navigation Distance Between Marine Routes...,2020-02-25,The purpose of this study was to determine the...,article,<NA>,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
2,10.3390/s22082857,Evaluation of the Effectiveness of Active and ...,2022-04-08,The risk of ship-bridge collisions should be e...,article,<NA>,https://app.dimensions.ai/details/publication/...,B12 Engineering,"Accidents, Traffic; Bayes Theorem; Computer Si..."
3,10.1016/j.heliyon.2024.e40965,A numerical investigation into the strength of...,2024-12-06,"In addition to the usual loads, fixed jacket o...",article,<NA>,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
4,10.1038/s41598-025-90132-x,The impact of underwater waves on ship manoeuv...,2025-02-15,"In 2019, the Motor Vessel Jaeger Arrow collide...",article,<NA>,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
...,...,...,...,...,...,...,...,...,...
1532,10.20944/preprints202505.0837.v6,Reintegrating Platonic Intimacy: A Literary an...,2025-05-29,"The present manuscript, rooted in literary rev...",preprint,<NA>,https://app.dimensions.ai/details/publication/...,D27 English Language and Literature,NaN
1533,10.20944/preprints202505.0837.v7,Reintegrating Platonic Intimacy: A Literary an...,2025-06-04,"The present manuscript, rooted in literary rev...",preprint,<NA>,https://app.dimensions.ai/details/publication/...,D31 Theology and Religious Studies,NaN
1534,10.20944/preprints202505.0837.v8,Reintegrating Platonic Intimacy: A Literary an...,2025-06-09,"The present manuscript, rooted in literary rev...",preprint,<NA>,https://app.dimensions.ai/details/publication/...,D31 Theology and Religious Studies,NaN
1535,10.5204/mcj.689,Double Quote Unquote: Scholarly Attribution as...,2013-08-12,"Many years ago, while studying in Paris as a n...",article,<NA>,https://app.dimensions.ai/details/publication/...,C23 Education,NaN


In [29]:
df_papers_03 = merge_papers(df_papers_02, df_dimensions, "dimensions")
df_papers_03

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,is_reference,...,relevance_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex,id_dimensions,topic_dimensions,keywords_dimensions
0,1 Probability of ship collision and grounding,10.1016/b978-0-12-815022-1.00001-3,book-chapter,<NA>,NaT,"To quantify the risk involved in ship traffic,...",NaN,NaN,<NA>,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
1,10th International Conference on Coastal and O...,NaN,other,<NA>,2024-01-01,The proceedings contain 28 papers. The special...,2-s2.0-85200940949,NaN,554,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886697517,NaN,361,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886707330,NaN,362,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"12th AIAA Aviation Technology, Integration, an...",NaN,other,<NA>,2012-12-01,The proceedings contain 306 papers. The specia...,2-s2.0-84880934476,NaN,1042,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3507,파일지지 구조물의 선박 충돌거동에 대한 해석,NaN,article,<NA>,2008-05-01,The ship collision analysis of steel pile grou...,NaN,NaN,<NA>,False,...,2.374983,https://content.openalex.org/works/W1548055064...,Engineering Applied Research,Engineering,Civil and Structural Engineering,Physical Sciences,Collision|Pier|Pile|Pile cap|Structural engine...,NaN,NaN,NaN
3508,평택·당진항의 내항개발에 따른 교통혼잡도 평가에 관한 연구,NaN,article,<NA>,2005-08-31,The port entry system of the inner harbor in P...,NaN,NaN,<NA>,False,...,1.518602,https://content.openalex.org/works/W3183424587...,Maritime Ports and Logistics,Engineering,Industrial and Manufacturing Engineering,Physical Sciences,Port (circuit theory)|Bridge (graph theory)|Ma...,NaN,NaN,NaN
3509,항만횡단 해상교량의 해양사고 관련 인자 분석을 통한 교량설계안 제안,NaN,article,<NA>,2008-12-31,Recently Bridges crossing waterway are constru...,NaN,NaN,<NA>,False,...,1.719204,https://content.openalex.org/works/W2476308561...,Marine and Coastal Research,Engineering,Ocean Engineering,Physical Sciences,Bridge (graph theory)|Span (engineering)|Mista...,NaN,NaN,NaN
3510,해상교량에 대한 선박충돌 위험도 평가 및 민감도 분석,NaN,article,<NA>,2013-09-01,In the design phase of sea-crossing bridge pro...,NaN,NaN,<NA>,False,...,2.922672,https://content.openalex.org/works/W2197667956...,Engineering Applied Research,Engineering,Civil and Structural Engineering,Physical Sciences,Collision|Bridge (graph theory)|Risk assessmen...,NaN,NaN,NaN


### Add useful columns

In [33]:
from slrrag.main import complete_papers_df, show_merged_stats
from slrrag.settings import PATH_CSV_01_QUERY_RESULTS

databases_names = ["scopus", "openalex", "dimensions"]
df_papers_04 = complete_papers_df(df_papers_03, database_names=databases_names)

df_papers_04.to_csv(PATH_CSV_01_QUERY_RESULTS)
print("Phase 1 results saved to :", PATH_CSV_01_QUERY_RESULTS)

---------CLEANING------------
Filtering 707 papers that are before year 2010
Filtering 371 papers that are not['article', 'conference-paper', 'book-chapter', 'book', 'conference-proceeding']
Phase 1 results saved to : /home/vlitoux/Documents/SLR-RAG/data/papers_selection/01_papers_query_results.csv


## Papers Analysis

In [34]:
# list of databases used
show_merged_stats(df_papers_04, databases_names)

---------SUMMARIZING------------
Total number of papers found: 2434
Reference papers found: 22 over 22
Number of common papers in all databases: 963
Number of papers in scopus: 1611
Number of reference papers in scopus: 19
Number of papers in openalex: 1746
Number of reference papers in openalex: 22
Number of papers in dimensions: 1412
Number of reference papers in dimensions: 18


#### Check which paper are not returned in the end

In [42]:
from slrrag.settings import PATH_INPUT


df_selected_papers_round_1 = pd.read_csv(PATH_INPUT / "nra_kept_papers_5.csv")
df_selected_papers_round_1["title_slug"] = df_selected_papers_round_1["title"].apply(
    slugify
)
df_selected_papers_round_1.head()

,file name,year,title,keywords,nra_lr_classification,nra_lr_perspective,nra_lr_refusal_resons,doi,title_slug
0,@zhangStudiesProbabilisticModel2012,2012,Studies on the Probabilistic Model for Ship-Br...,"[[Literature review]], [[Ship-Bridge Allusion]...",[[Frequency]],[[Strategic Navigational Risk Assessment (SNRA)]],NaN,10.5539/ijsp.v2n1p16,studies-on-the-probabilistic-model-for-ship-br...
1,@daiRiskCollisionService2013,2013,Risk of collision between service vessels and ...,"[[Risk-Influencing Factors (RIFs)]], [[Fault t...","[[Risk Causation]], [[Frequency]]",[[Strategic Navigational Risk Assessment (SNRA)]],NaN,10.1016/j.ress.2012.07.008,risk-of-collision-between-service-vessels-and-...
2,@zhengNovelShipBridgeCollision2013,2013,A Novel Ship-Bridge Collision Avoidance System...,"[[Ship-Bridge Allusion]], [[Bridge (infrastruc...",Alert System,[[Operational Navigational Risk Assessment (ON...,NaN,10.19026/rjaset.6.4176,a-novel-ship-bridge-collision-avoidance-system...
3,@zhouNanjingYangtzeRiver2013,2013,Nanjing Yangtze River Bridge Transit Capacity ...,"[[Ship domain]], [[Poisson Distribution]], [[Q...",Traffic Flow density,[[Operational Navigational Risk Assessment (ON...,NaN,10.1016/j.sbspro.2013.08.285,nanjing-yangtze-river-bridge-transit-capacity-...
4,@jilunEvaluationRiskShip2014,2014,Evaluation on Risk for Ship Collision with Arc...,"[[Bridge (infrastructure)]], [[Risk matrix]], ...",[[Frequency]],[[Strategic Navigational Risk Assessment (SNRA)]],NaN,10.2174/1874149501408010351,evaluation-on-risk-for-ship-collision-with-arc...


In [45]:
df_papers_03[df_papers_03["title"].str.lower().str.contains("restricted waters")]

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,is_reference,...,relevance_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex,id_dimensions,topic_dimensions,keywords_dimensions
2286,Pre-warning system analysis on dynamic risk of...,10.1109/ICTIS.2017.8047843,conference-paper,<NA>,2017-09-20,Risk pre-warning of ship collision with bridge...,2-s2.0-85032804229,fuzzy inference system(FIS) | marine traffic s...,260,False,...,93.369965,https://content.openalex.org/works/W2758184234...,Structural Integrity and Reliability Analysis,Engineering,Mechanical Engineering,Physical Sciences,Bridge (graph theory)|Warning system|Early war...,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN


In [43]:
print(
    "\n".join(
        list(
            df_selected_papers_round_1[
                ~(
                    df_selected_papers_round_1["title_slug"].isin(
                        df_papers_04["title_slug"]
                    )
                )
            ]["title"].values
        )
    )
)

Risk assessment of ship navigation in Bridge Areas
Safety Zones Around Offshore Installations
Featured risk evaluation of nautical navigational environment using a risk cloud model
A study on influence of Putian Port offshore wind farm construction on navigation safety
A novel fuzzy Bayesian network-based MADM model for offshore wind turbine selection in busy waterways: An application to a case in China
Proposal of new measures for risk assessment in navigation – a case study of the M/V Cosco Busan accident
Analysis of navigational risk indicators as a function of the ship's domain width for the selected offshore wind farm in the Baltic Sea
Risk assessment of general FPSO supply system based on hybrid fuzzy fault tree and Bayesian network
Comparison of Fuzzy Bayesian Network Methods for Maritime Applications
A Monocular Vision-Based Safety Monitoring Framework for Offshore Infrastructures Utilizing Grounded SAM
There's no such thing as free shipping: The significant risk large vessels 

#### Upset plot

In [36]:
from slrrag.settings import PATH_UPSET
from slrrag.upset_plotter import plot_upset


path_upset = PATH_UPSET
fig_upset = plot_upset(df_papers_04)
fig_upset.write_html(path_upset)
print("Upset plot saved to : ", path_upset)
fig_upset

Upset plot saved to :  /home/vlitoux/Documents/SLR-RAG/data/papers_selection/upset.html


## Old : with lens

In [ ]:
from slrrag.main import print_lens_export_info
from slrrag.query_converter import convert_query_to_lens
from slrrag.settings import PATH_LENS_EXPORT

lens_export_file = PATH_LENS_EXPORT
lens_query = convert_query_to_lens(query_test_for_lens)
print_lens_export_info(lens_query, lens_export_file)

In [ ]:
from slrrag.querier_lens import read_lens_export

df_lens = read_lens_export(lens_export_file)
df_lens

In [ ]:
df_papers = merge_papers(df_papers, df_lens, "lens")
df_papers